# Demo 10e — The benchmark grid

**Class 10 · E1** (slides: `slides/10_clustering.md`)

Seven methods on six toy sets, one picture and one score board:
$k$-means, Ward, single linkage, DBSCAN, GMM, fuzzy $c$-means and NMF (soft memberships of a non-negative lift).
Every method gets the true $k$ where it takes one; DBSCAN gets its $\varepsilon$ from the knee of the $k$-distance
graph. The score is the adjusted Rand index (ARI): 1 = the generating groups, 0 = chance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from kneeliverse import kneedle
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score as ari
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

from faa import clustering as cl
from faa import demo, plotlib
from faa.demo import show, slide

demo.setup("10")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (plotlib.BLUE, plotlib.ORANGE, plotlib.GREEN, plotlib.RED, plotlib.SKY, plotlib.PURPLE, plotlib.GREY)
COLS = [BLUE, ORANGE, GREEN, RED, SKY, PURPLE]
SEED = 0
METHODS = ["k-means", "Ward", "single", "DBSCAN", "GMM", "FCM", "NMF"]
SETS = list(cl.BENCH)
B = cl.benchmarks(SEED)

In [ ]:
def knee_eps(X, m=5):
    """eps from the knee of the sorted distance to the m-th neighbour (Kneedle)."""
    d = np.sort(NearestNeighbors(n_neighbors=m).fit(X).kneighbors(X)[0][:, -1])
    return float(d[kneedle.knee(np.c_[np.arange(len(d)), d])])


def run(X, k, seed=SEED):
    """Labels of the seven methods on X; `uniform` has no groups, k = 3 is only a request."""
    Xc = X - X.mean(0)
    lift = np.c_[np.maximum(Xc, 0), np.maximum(-Xc, 0)]
    _, Uf, _ = cl.fcm(X, k, seed=seed)
    gm = GaussianMixture(k, covariance_type="full", n_init=3, random_state=seed).fit(X)
    _, Wn, Hn, _ = pynnmf.nmf_hals(lift.T, k, n=300, seed=seed, init="random")
    out = {
        "k-means": KMeans(k, n_init=10, random_state=seed).fit_predict(X),
        "Ward": AgglomerativeClustering(k, linkage="ward").fit_predict(X),
        "single": AgglomerativeClustering(k, linkage="single").fit_predict(X),
        "DBSCAN": DBSCAN(eps=knee_eps(X), min_samples=5).fit_predict(X),
        "GMM": gm.predict(X),
        "FCM": Uf.argmax(1),
        "NMF": cl.nmf_membership(Wn, Hn).argmax(1),
    }
    soft = {"GMM": gm.predict_proba(X), "FCM": Uf, "NMF": cl.nmf_membership(Wn, Hn)}
    return out, soft

## Slide: Seven methods, six sets

One row per data set, one column per method. Grey points are DBSCAN noise. Panels of the soft methods show the
membership of each point by the size of the marker (large = certain).

In [ ]:
labels, soft, score = {}, {}, {}
for name in SETS:
    X, y = B[name]
    k = 3 if name == "uniform" else len(set(y))
    labels[name], soft[name] = run(X, k)
    score[name] = {m: ari(y, labels[name][m]) for m in METHODS}

f, ax = plt.subplots(len(SETS), len(METHODS), figsize=(plotlib.FULL, 2.9), sharex="row", sharey="row")
for i, name in enumerate(SETS):
    X, y = B[name]
    for j, m in enumerate(METHODS):
        lab = labels[name][m]
        size = 3.0
        if m in soft[name]:
            size = 1.5 + 4 * soft[name][m].max(1)
        ax[i, j].scatter(*X.T, c=[GREY if v < 0 else COLS[v % 6] for v in lab], s=size, lw=0)
        ax[i, j].set_xticks([]), ax[i, j].set_yticks([])
        ax[i, j].grid(False)
        if i == 0:
            ax[i, j].set_title(m, fontsize=7)
        if j == 0:
            ax[i, j].set_ylabel(name, fontsize=7)
        ax[i, j].text(0.04, 0.04, f"{score[name][m]:.2f}", transform=ax[i, j].transAxes, fontsize=6.5, bbox={"fc": "white", "ec": "none", "pad": 0.5, "alpha": 0.8})
f.subplots_adjust(left=0.06, right=0.995, top=0.96, bottom=0.01, wspace=0.05, hspace=0.08)
plotlib.save(f, "e1_grid")
slide("Seven methods, six sets")
board = pl.DataFrame([(n, *[score[n][m] for m in METHODS]) for n in SETS], schema=["set", *METHODS], orient="row")
show(board)
print("mean ARI over the five sets with groups:", {m: round(float(np.mean([score[n][m] for n in SETS[:5]])), 3) for m in METHODS})

## Slide: Who wins where

The score board as a heat map. Each row has its own winner: no method is best on all six sets (the uniform row has
no truth: every score is near 0).

In [ ]:
M = np.array([[score[n][m] for m in METHODS] for n in SETS])
f, ax = plt.subplots(figsize=(plotlib.FULL, 1.9))
im = ax.imshow(np.clip(M, 0, 1), cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(METHODS)), METHODS), ax.set_yticks(range(len(SETS)), SETS)
ax.grid(False)
for i in range(len(SETS)):
    for j in range(len(METHODS)):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color="w" if M[i, j] < 0.6 else "k", fontweight="bold" if M[i, j] == M[i].max() else "normal")
f.colorbar(im, ax=ax, label="ARI", shrink=0.8)
plotlib.save(f, "e1_scoreboard")
slide("Who wins where")
print({n: [METHODS[j] for j in np.flatnonzero(M[i] >= M[i].max() - 1e-9)] for i, n in enumerate(SETS[:5])})

## Sweep: seeds and sample size

Ten seeds of every generator (new data, new starts) on the sets where several methods tie: the mean and sd of the
ARI. A winner by 0.01 on one seed is not a winner.

In [ ]:
rows = []
for name in ("blobs", "aniso", "varied", "moons", "circles"):
    runs = []
    for s in range(10):
        Bs = cl.benchmarks(s)
        X, y = Bs[name]
        lab, _ = run(X, len(set(y)), seed=s)
        runs.append([ari(y, lab[m]) for m in METHODS])
    runs = np.array(runs)
    rows.append((name, *[f"{a:.2f} ± {b:.2f}" for a, b in zip(runs.mean(0), runs.std(0))]))
slide("Ten seeds")
show(pl.DataFrame(rows, schema=["set", *METHODS], orient="row"))

## Failure: noise and scale

Add 10 % uniform background noise to the blobs: DBSCAN labels it as noise, the others assign it to a cluster.
And rescale one feature by 100: every distance-based method is dominated by that feature.

In [ ]:
X, y = B["blobs"]
rng = np.random.default_rng(SEED)
Xn = np.r_[X, rng.uniform(X.min(0) - 2, X.max(0) + 2, (30, 2))]
yn = np.r_[y, np.full(30, -1)]
keep = yn >= 0
labn, _ = run(Xn, 3)
Xs = X * np.array([1.0, 100.0])
labs, _ = run(Xs, 3)
f, ax = plt.subplots(figsize=(plotlib.FULL, 1.5))
w = 0.27
for q, (title, vals, c) in enumerate((("clean", [ari(y, labels["blobs"][m]) for m in METHODS], BLUE),
                                      ("10% noise added", [ari(yn[keep], labn[m][keep]) for m in METHODS], ORANGE),
                                      ("feature 2 x 100", [ari(y, labs[m]) for m in METHODS], RED))):
    ax.bar(np.arange(len(METHODS)) + (q - 1) * w, vals, w, color=c, label=title)
ax.set_xticks(range(len(METHODS)), METHODS), ax.set_ylabel("ARI"), ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18))
plotlib.save(f, "e1_noise_scale")
slide("Noise and scale")
show(pl.DataFrame([(m, ari(y, labels["blobs"][m]), ari(yn[keep], labn[m][keep]), ari(y, labs[m])) for m in METHODS],
                  schema=["method", "clean", "10% noise added", "feature 2 x 100"], orient="row"))
print(f"DBSCAN flags {(labn['DBSCAN'][~keep] == -1).sum()} of the 30 noise points; k-means flags none")

## Slide: Strengths and limits as strips

For each method of the class, the same three sets side by side (blobs: works, stretched, moons: the usual failures),
with the ARI of the grid above: the picture behind the "strengths and limits" frames.

In [ ]:
STRIPS = {
    "kmeans": (("k-means", "blobs"), ("k-means", "aniso"), ("k-means", "moons")),
    "hier": (("Ward", "blobs"), ("single", "moons"), ("single", "blobs")),
    "dbscan": (("DBSCAN", "moons"), ("DBSCAN", "circles"), ("DBSCAN", "varied")),
    "gmm": (("GMM", "aniso"), ("GMM", "varied"), ("GMM", "moons")),
    "fcmnmf": (("FCM", "blobs"), ("FCM", "aniso"), ("NMF", "blobs"), ("NMF", "aniso")),
}


def strip(cells):
    """One row of panels: method, set and ARI of the grid above."""
    f, ax = plt.subplots(1, len(cells), figsize=(plotlib.FULL, 1.2))
    for a, (m, name) in zip(ax, cells):
        X, y = B[name]
        a.scatter(*X.T, c=[GREY if v < 0 else COLS[v % 6] for v in labels[name][m]], s=3, lw=0)
        a.set_title(f"{m}, {name}: {score[name][m]:.2f}", fontsize=7)
        a.set_xticks([]), a.set_yticks([]), a.grid(False)
    return f


plotlib.save(strip(STRIPS["kmeans"]), "e1_strip_kmeans")
plotlib.save(strip(STRIPS["hier"]), "e1_strip_hier")
plotlib.save(strip(STRIPS["dbscan"]), "e1_strip_dbscan")
plotlib.save(strip(STRIPS["gmm"]), "e1_strip_gmm")
plotlib.save(strip(STRIPS["fcmnmf"]), "e1_strip_fcmnmf")
slide("Strengths and limits as strips")
print({k: [f"{m}/{n} {score[n][m]:.2f}" for m, n in cells] for k, cells in STRIPS.items()})

## Figure: ten seeds

Mean and sd of the ARI over ten draws, three shapes.

In [ ]:
f, ax = plt.subplots(figsize=(plotlib.FULL, 1.5))
shapes = ("aniso", "moons", "circles")
w = 0.11
for q_, name in enumerate(shapes):
    runs = []
    for s_ in range(10):
        Xr, yr = cl.benchmarks(s_)[name]
        lab_, _ = run(Xr, len(set(yr)), seed=s_)
        runs.append([ari(yr, lab_[m]) for m in METHODS])
    runs = np.array(runs)
    ax.errorbar(np.arange(len(METHODS)) + (q_ - 1) * 0.2, runs.mean(0), yerr=runs.std(0), fmt="o", ms=3, lw=0.8, color=COLS[q_], label=name)
ax.set_xticks(range(len(METHODS)), METHODS), ax.set_ylabel("ARI"), ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.2))
plotlib.save(f, "e1_seeds")